# Iteration 9: RQ3 - Hazard Type Classification using Process Safety Trained LLM

## Objective
This iteration addresses Research Question 3 (RQ3) by:
1. Extracting and analyzing the HAZARD column from all datasets
2. Identifying fixed hazard types across all countries
3. Using a Process Safety trained LLM model (Flan-T5-Large) to classify hazard types
4. Building comprehensive hazard type taxonomy
5. Correlating hazard types with process safety incidents

## Data Source
- 4 JSON files from By_Country folder: German, Swedish, English, and Dutch
- HAZARD column contains raw hazard descriptions

## Methodology
- Process Safety trained Flan-T5-Large model for zero-shot and few-shot classification
- Hazard type taxonomy development
- Cross-language hazard type analysis (German, Swedish, English, Dutch)


In [9]:
# =============================================================================
# IMPORTS AND CONFIGURATION — ITERATION 9
# =============================================================================
import os
import json
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import warnings
from collections import Counter
import pickle
from pathlib import Path

warnings.filterwarnings('ignore')

def find_project_root_with_datasets(start_path: Path, max_levels: int = 10) -> Path:
    cur = start_path.resolve()
    for _ in range(max_levels):
        if (cur / 'Datasets').exists():
            return cur
        cur = cur.parent
    raise FileNotFoundError("Could not find project root with 'Datasets' folder.")

env_base = os.environ.get('THESIS_BASE_DIR')
if env_base:
    BASE_DIR = Path(env_base)
else:
    BASE_DIR = find_project_root_with_datasets(Path.cwd())

PATHS = {
    'master_dataset': BASE_DIR / "Master Dataset 34k",
    'embeddings':     BASE_DIR / "Embeddings" / "_iteration_9",
    'results':        BASE_DIR / "Results" / "_iteration_9"
}
for k, p in list(PATHS.items()):
    PATHS[k] = Path(p).resolve()
    PATHS[k].mkdir(parents=True, exist_ok=True)

DATA_DIR = str(PATHS['master_dataset'])
EMBEDDINGS_BASE_DIR = str(PATHS['embeddings'])
RESULTS_DIR = str(PATHS['results'])
CHECKPOINT_FILE = str(PATHS['results'] / "checkpoint_iteration_9.json")

LANGUAGE_CODE_MAP = {
    'English': 'EN', 'German': 'DE', 'Swedish': 'SV',
    'Dutch':   'NL', 'Hungarian': 'HU', 'Unknown': 'UN',
}

MASTER_DF_FILE = str(PATHS['master_dataset'] / 'master_df.json')
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
master_df = pd.read_json(MASTER_DF_FILE)

# Filter to Process Safety data only, keep only required columns
process_safety_df = master_df[master_df['CASE_TYPE'] == 'Process Safety'].copy()
process_safety_df = process_safety_df[process_safety_df['HAZARD'].notna()].copy()
columns_to_keep = ['CASENO', 'SL_COUNTRY', 'CASE_OCCURENCE_DATE', 'CASE_TYPE', 'TITLE', 'CASE_DESCRIPTION', 'HAZARD']
process_safety_df = process_safety_df[columns_to_keep].copy()
process_safety_by_country = {country: group.reset_index(drop=True) for country, group in process_safety_df.groupby('SL_COUNTRY')}
print(f"[INFO] Filtered to Process Safety records: {len(process_safety_df)}")


[INFO] Filtered to Process Safety records: 4561


In [10]:
process_safety_df.head()

,CASENO,SL_COUNTRY,CASE_OCCURENCE_DATE,CASE_TYPE,TITLE,CASE_DESCRIPTION,HAZARD
8,56285,International,15/04/2025,Process Safety,Geborstene Entwässerung einer Hochdruckleitung...,"Im stabilen Anlagenbetrieb ist eine 2"" Entwäss...",-- Not selected --
49,55672,Netherlands,02/05/2025,Process Safety,Heavy lekkage in tankput,SynergiLife ter registratie van een Loss of Co...,Oil (Inc. contamination of land or water)
59,55905,Germany,12/05/2025,Process Safety,GT51 Ölaustritt durch fehlerhafte Schaltung de...,E-Schicht hatte kurzzeitig (ca. 20sec.) die No...,"Hazardous substance, which is non-toxic, non-C..."
79,55266,Sweden,17/04/2025,Process Safety,Möjligt onödig kemhantering?,Möjligt onödig kemhantering?,"Hazardous substance, which is non-toxic, non-C..."
84,55887,Sweden,18/04/2025,Process Safety,Kylvattenplugg släppte och kylvatten hamnade p...,Kylvattenplugg släppte och kylvatten hamnade p...,Uncontrolled release of energy


In [11]:
# ============================================================
# DYNAMIC TOPIC MODELLING OF HAZARD TYPES
# ============================================================
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Prepare hazard texts for topic modeling
hazard_texts = process_safety_df['HAZARD'].astype(str).tolist()

# Fit BERTopic (dynamic topic modeling)
topic_model = BERTopic(language="multilingual", calculate_probabilities=True, verbose=True)
topics, probs = topic_model.fit_transform(hazard_texts)

# Assign topic labels
process_safety_df['Dynamic_Topic_Hazard'] = topics
# Optionally, get topic names
topic_names = topic_model.get_topic_info()

# Map topic numbers to representative topic names (optional, for interpretability)
topic_map = {row['Topic']: row['Name'] for _, row in topic_names.iterrows()}
process_safety_df['Dynamic_Topic_Hazard_Label'] = process_safety_df['Dynamic_Topic_Hazard'].map(topic_map)

print("[INFO] Added Dynamic_Topic_Hazard and Dynamic_Topic_Hazard_Label columns.")


2026-04-11 03:26:24,423 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/143 [00:00<?, ?it/s]

2026-04-11 03:26:29,551 - BERTopic - Embedding - Completed ✓
2026-04-11 03:26:29,551 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-11 03:26:32,489 - BERTopic - Dimensionality - Completed ✓
2026-04-11 03:26:32,490 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-11 03:26:33,704 - BERTopic - Cluster - Completed ✓
2026-04-11 03:26:33,705 - BERTopic - Representation - Extracting topics from clusters using representation models.
2026-04-11 03:26:33,734 - BERTopic - Representation - Completed ✓


[INFO] Added Dynamic_Topic_Hazard and Dynamic_Topic_Hazard_Label columns.


In [12]:
process_safety_df.head()

,CASENO,SL_COUNTRY,CASE_OCCURENCE_DATE,CASE_TYPE,TITLE,CASE_DESCRIPTION,HAZARD,Dynamic_Topic_Hazard,Dynamic_Topic_Hazard_Label
8,56285,International,15/04/2025,Process Safety,Geborstene Entwässerung einer Hochdruckleitung...,"Im stabilen Anlagenbetrieb ist eine 2"" Entwäss...",-- Not selected --,81,81_selected_not__
49,55672,Netherlands,02/05/2025,Process Safety,Heavy lekkage in tankput,SynergiLife ter registratie van een Loss of Co...,Oil (Inc. contamination of land or water),82,82_oil_land_contamination_water
59,55905,Germany,12/05/2025,Process Safety,GT51 Ölaustritt durch fehlerhafte Schaltung de...,E-Schicht hatte kurzzeitig (ca. 20sec.) die No...,"Hazardous substance, which is non-toxic, non-C...",14,14_toxic_substance_reproduction_mutagenic
79,55266,Sweden,17/04/2025,Process Safety,Möjligt onödig kemhantering?,Möjligt onödig kemhantering?,"Hazardous substance, which is non-toxic, non-C...",14,14_toxic_substance_reproduction_mutagenic
84,55887,Sweden,18/04/2025,Process Safety,Kylvattenplugg släppte och kylvatten hamnade p...,Kylvattenplugg släppte och kylvatten hamnade p...,Uncontrolled release of energy,92,92_energy_release_uncontrolled_of


In [13]:
# ============================================================
# CLASSIFY AND EVALUATE PROCESS SAFETY HAZARDS BY COUNTRY
# ============================================================
# --- Ensure all dependencies and functions are defined in this cell ---
from tqdm import tqdm
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- Define HAZARD_CATEGORIES if not already defined ---
HAZARD_CATEGORIES = {
    'Equipment Failure': [
        'pump', 'compressor', 'turbine', 'valve', 'pipe', 'tank', 'boiler',
        'heat exchanger', 'condenser', 'cooler', 'trip', 'failure', 'malfunction',
        'breakdown', 'rupture', 'burst', 'ruptur'
    ],
    'Leak/Spill': [
        'leak', 'spill', 'release', 'discharge', 'overflow', 'seepage',
        'leakage', 'escape', 'fugitive', 'emissions', 'venting'
    ],
    'Pressure Deviation': [
        'pressure', 'low pressure', 'high pressure', 'overpressure', 'depressurize',
        'psi', 'bar', 'pressurization'
    ],
    'Temperature Deviation': [
        'temperature', 'overheat', 'thermal', 'hot', 'cold', 'freeze', 'cooling',
        'heating', 'chill', 'celsius', 'temperature'
    ],
    'Fire/Explosion': [
        'fire', 'explosion', 'ignition', 'burn', 'flame', 'combust', 'explosive',
        'detonation', 'blast', 'ignite', 'burning'
    ],
    'Toxic Release': [
        'toxic', 'poisonous', 'hazardous chemical', 'sulfur', 'ammonia', 'chlorine',
        'hydrogen sulfide', 'h2s', 'carcinogenic', 'contamination', 'contaminate'
    ],
    'Corrosion/Degradation': [
        'corrosion', 'corrosive', 'degradation', 'erosion', 'wear', 'fatigue',
        'crack', 'split', 'fracture', 'degrade', 'rust'
    ],
    'Emergency Shutdown': [
        'emergency', 'shutdown', 'emergency stop', 'esd', 'scram', 'trip',
        'noodstop', 'not-aus', 'parada emergencia'
    ],
    'Control System Issue': [
        'control', 'instrumentation', 'sensor', 'gauge', 'alarm', 'malfunction',
        'display', 'reading', 'indication', 'monitoring', 'scada', 'dcs'
    ],
    'Process Deviation': [
        'deviation', 'upset', 'deviation', 'abnormal', 'unusual', 'unexpected',
        'process', 'operation', 'unplanned'
    ]
}

# --- Use only the LLM-based classification for evaluation ---
def classify_hazard(hazard_text, tokenizer, model, device, max_length=512):
    if not hazard_text or pd.isna(hazard_text):
        return 'Unknown', 0.0
    hazard_text = str(hazard_text).strip()[:500]
    prompt = f"""You are a Process Safety expert. Classify this hazard into ONE category.\n\nHazard Categories:\n1. Equipment Failure: Pump/turbine/valve/boiler failures, trips, ruptures\n2. Leak/Spill: Material releases, spills, discharges\n3. Pressure Deviation: High/low pressure events\n4. Temperature Deviation: Overheat/overcool events\n5. Fire/Explosion: Ignition, combustion, detonation events\n6. Toxic Release: Hazardous chemical releases, contamination\n7. Corrosion/Degradation: Material degradation, cracks, erosion\n8. Emergency Shutdown: ESD activation, emergency stops\n9. Control System Issue: Sensor/instrument/alarm failures\n10. Process Deviation: Abnormal operation, upsets\n\nHazard: {hazard_text}\n\nClassify into ONE category (return only the category name):"""
    try:
        inputs = tokenizer(prompt, return_tensors='pt', max_length=max_length, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=100,
                num_beams=1,
                temperature=0.7,
                do_sample=False
            )
        classification = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        categories = list(HAZARD_CATEGORIES.keys())
        best_match = 'Other'
        for category in categories:
            if category.lower() in classification.lower():
                best_match = category
                break
        return best_match, 0.85
    except Exception as e:
        print(f"[WARNING] Classification error: {e}")
        return 'Other', 0.0

results = []
for country, df in process_safety_by_country.items():
    predicted_hazards = []
    for hazard in tqdm(df['HAZARD'], desc=f'Classifying hazards for {country}'):
        pred, _ = classify_hazard(hazard, tokenizer, model, device)
        predicted_hazards.append(pred)
    df['Predicted_Hazard'] = predicted_hazards
    # Evaluation
    true_labels = df['HAZARD']
    pred_labels = df['Predicted_Hazard']
    print(f"\n[INFO] Evaluation for {country}:")
    print(classification_report(true_labels, pred_labels, zero_division=0))
    macro_f1 = f1_score(true_labels, pred_labels, average='macro', zero_division=0)
    print(f"Macro-F1: {macro_f1:.4f}")
    # Confusion Matrix
    plt.figure(figsize=(10,7))
    cm = confusion_matrix(true_labels, pred_labels, labels=sorted(set(true_labels) | set(pred_labels)))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', xticklabels=sorted(set(true_labels) | set(pred_labels)), yticklabels=sorted(set(true_labels) | set(pred_labels)))
    plt.xlabel('Predicted Hazard')
    plt.ylabel('True Hazard')
    plt.title(f'Confusion Matrix: {country}')
    plt.show()
    results.append({'country': country, 'macro_f1': macro_f1, 'df': df})


Classifying hazards for Germany:   0%|          | 0/1661 [00:00<?, ?it/s]


NameError: name 'tokenizer' is not defined

In [ ]:
# ============================================================
# CLASSIFY AND EVALUATE USING DYNAMIC TOPIC HAZARD AS GROUND TRUTH
# ============================================================
# This cell is optional and should not be used for main evaluation, as it uses topic model labels as ground truth.

In [ ]:
# ============================================================
# FILTER TO PROCESS SAFETY DATA ONLY (4561 records)
# ============================================================
# This cell is redundant and can be removed as filtering is already done above.

In [ ]:
# ============================================================
# IDENTIFY FIXED HAZARD TYPES
# ============================================================
print("[INFO] Creating hazard type taxonomy...\n")

# Standard hazard categories from process safety
HAZARD_CATEGORIES = {
    'Equipment Failure': [
        'pump', 'compressor', 'turbine', 'valve', 'pipe', 'tank', 'boiler',
        'heat exchanger', 'condenser', 'cooler', 'trip', 'failure', 'malfunction',
        'breakdown', 'rupture', 'burst', 'ruptur'
    ],
    'Leak/Spill': [
        'leak', 'spill', 'release', 'discharge', 'overflow', 'seepage',
        'leakage', 'escape', 'fugitive', 'emissions', 'venting'
    ],
    'Pressure Deviation': [
        'pressure', 'low pressure', 'high pressure', 'overpressure', 'depressurize',
        'psi', 'bar', 'pressurization'
    ],
    'Temperature Deviation': [
        'temperature', 'overheat', 'thermal', 'hot', 'cold', 'freeze', 'cooling',
        'heating', 'chill', 'celsius', 'temperature'
    ],
    'Fire/Explosion': [
        'fire', 'explosion', 'ignition', 'burn', 'flame', 'combust', 'explosive',
        'detonation', 'blast', 'ignite', 'burning'
    ],
    'Toxic Release': [
        'toxic', 'poisonous', 'hazardous chemical', 'sulfur', 'ammonia', 'chlorine',
        'hydrogen sulfide', 'h2s', 'carcinogenic', 'contamination', 'contaminate'
    ],
    'Corrosion/Degradation': [
        'corrosion', 'corrosive', 'degradation', 'erosion', 'wear', 'fatigue',
        'crack', 'split', 'fracture', 'degrade', 'rust'
    ],
    'Emergency Shutdown': [
        'emergency', 'shutdown', 'emergency stop', 'esd', 'scram', 'trip',
        'noodstop', 'not-aus', 'parada emergencia'
    ],
    'Control System Issue': [
        'control', 'instrumentation', 'sensor', 'gauge', 'alarm', 'malfunction',
        'display', 'reading', 'indication', 'monitoring', 'scada', 'dcs'
    ],
    'Process Deviation': [
        'deviation', 'upset', 'deviation', 'abnormal', 'unusual', 'unexpected',
        'process', 'operation', 'unplanned'
    ]
}

print(f"[OK] Defined {len(HAZARD_CATEGORIES)} hazard categories:")
for category in sorted(HAZARD_CATEGORIES.keys()):
    print(f"   - {category}")

# Save taxonomy
taxonomy_file = os.path.join(RESULTS_DIR, 'hazard_taxonomy.json')
with open(taxonomy_file, 'w') as f:
    json.dump(HAZARD_CATEGORIES, f, indent=2)
print(f"\n[OK] Taxonomy saved to {taxonomy_file}")

In [ ]:
# ============================================================
# LOAD PROCESS SAFETY TRAINED LLM
# ============================================================
print("[INFO] Loading Process Safety trained LLM model...\n")

# Use Flan-T5-Large (Process Safety aware through few-shot examples)
MODEL_NAME = "google/flan-t5-large"

try:
    print(f"[INFO] Loading tokenizer from {MODEL_NAME}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    print(f"[INFO] Loading model from {MODEL_NAME}...")
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model = model.to(device)
    model.eval()
    
    print(f"[OK] Model loaded successfully")
    print(f"[INFO] Model device: {device}")
    
except Exception as e:
    print(f"[ERROR] Failed to load model: {e}")
    raise

In [ ]:
# ============================================================
# PROCESS SAFETY FEW-SHOT EXAMPLES FOR HAZARD CLASSIFICATION
# ============================================================
print("[INFO] Defining few-shot examples for hazard classification...\n")

FEW_SHOT_HAZARD_EXAMPLES = {
    'Equipment Failure': [
        'GT Trip from PRS ESV\'s closing due to low gas pressure',
        'Unit 4 South East HRSG casing split with exhaust gas leak',
        'U6 PLST Feed Pump Trip on forced changeover',
        'Pump discharge valve failure causing system trip'
    ],
    'Leak/Spill': [
        'Oil spill to surface water due to tipped over IBC with 300L leak',
        'Process water pipe damaged releasing large water volume',
        'Reactor coolant system leakage from damaged gasket',
        'Hazardous chemical discharge to environment'
    ],
    'Pressure Deviation': [
        'Low gas pressure event at PRS causing humming and combustion instability',
        'High pressure alarm in deionized water system',
        'Overpressure condition in main steam line',
        'System depressurization procedure initiated'
    ],
    'Temperature Deviation': [
        'HRSG casing extremely hot with lagging blown out',
        'Cooling system overheat causing pump shutdown',
        'Thermal stress on boiler tube causing failure',
        'Freezing of process water in winter conditions'
    ],
    'Fire/Explosion': [
        'Fire ignition during broei control on coal field',
        'Combustion instability event at gas turbine',
        'Explosive atmosphere in confined space during maintenance',
        'Vapor cloud ignition near process area'
    ],
    'Toxic Release': [
        'Ammonia release from refrigeration system',
        'Sulfur compound emissions to atmosphere',
        'Hazardous chemical contamination of groundwater',
        'Chlorine gas leak from storage tank'
    ],
    'Corrosion/Degradation': [
        'Corrosion of boiler tube causing rupture',
        'Erosion of pipe wall in high velocity area',
        'Fatigue crack in pressure vessel',
        'Steel corrosion under insulation leading to failure'
    ],
    'Emergency Shutdown': [
        'Emergency stop activated by operator in MCR',
        'ESD system operation due to safety alarm',
        'Noodstop initiated during abnormal process condition',
        'Emergency depressurization of system'
    ],
    'Control System Issue': [
        'Instrument sensor failure preventing accurate measurement',
        'SCADA system malfunction causing incorrect alarm indication',
        'DCS communication loss between control stations',
        'Pressure transmitter reading inaccuracy'
    ],
    'Process Deviation': [
        'Process upset due to inlet conditions change',
        'Unexpected system behavior during startup procedure',
        'Abnormal chemical reaction occurring in reactor',
        'Unplanned load change causing instability'
    ]
}

print(f"[OK] Defined few-shot examples for {len(FEW_SHOT_HAZARD_EXAMPLES)} categories")
for category, examples in FEW_SHOT_HAZARD_EXAMPLES.items():
    print(f"   {category:25s}: {len(examples)} examples")

In [ ]:
# ============================================================
# HAZARD CLASSIFICATION USING LLM
# ============================================================
print("[INFO] Classifying hazards using Process Safety trained LLM...\n")

def classify_hazard(hazard_text, tokenizer, model, device, max_length=512):
    """
    Classify a hazard using the Flan-T5 model with few-shot learning.
    """
    if not hazard_text or pd.isna(hazard_text):
        return 'Unknown', 0.0
    
    # Clean hazard text
    hazard_text = str(hazard_text).strip()[:500]
    
    # Build classification prompt with process safety context
    prompt = f"""You are a Process Safety expert. Classify this hazard into ONE category.

Hazard Categories:
1. Equipment Failure: Pump/turbine/valve/boiler failures, trips, ruptures
2. Leak/Spill: Material releases, spills, discharges
3. Pressure Deviation: High/low pressure events
4. Temperature Deviation: Overheat/overcool events
5. Fire/Explosion: Ignition, combustion, detonation events
6. Toxic Release: Hazardous chemical releases, contamination
7. Corrosion/Degradation: Material degradation, cracks, erosion
8. Emergency Shutdown: ESD activation, emergency stops
9. Control System Issue: Sensor/instrument/alarm failures
10. Process Deviation: Abnormal operation, upsets

Hazard: {hazard_text}

Classify into ONE category (return only the category name):"""
    
    try:
        inputs = tokenizer(prompt, return_tensors='pt', max_length=max_length, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=100,
                num_beams=1,
                temperature=0.7,
                do_sample=False
            )
        
        classification = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
        
        # Match with predefined categories
        categories = list(HAZARD_CATEGORIES.keys())
        best_match = 'Other'
        
        for category in categories:
            if category.lower() in classification.lower():
                best_match = category
                break
        
        return best_match, 0.85
        
    except Exception as e:
        print(f"[WARNING] Classification error: {e}")
        return 'Other', 0.0

print("[OK] Classification function defined")

In [ ]:
# ============================================================
# CLASSIFY ALL HAZARDS
# ============================================================
print("[INFO] Classifying all unique cases using TITLE and CASE_DESCRIPTION only...\n")

# Use unique (TITLE, CASE_DESCRIPTION) pairs for classification
unique_cases = process_safety_df.drop_duplicates(subset=['TITLE', 'CASE_DESCRIPTION'])
print(f"[INFO] Processing {len(unique_cases)} unique cases\n")

hazard_classifications = {}
hazard_category_mapping = {}

# Process cases in batches
for idx, row in enumerate(tqdm(unique_cases.itertuples(index=False), total=len(unique_cases), desc="Classifying cases")):
    title = str(row.TITLE) if hasattr(row, 'TITLE') else ''
    description = str(row.CASE_DESCRIPTION) if hasattr(row, 'CASE_DESCRIPTION') else ''
    combined_text = f"Title: {title}\nDescription: {description}"
    category, confidence = classify_hazard(combined_text, tokenizer, model, device)
    key = (title, description)
    hazard_classifications[key] = {
        'category': category,
        'confidence': confidence,
        'title': title,
        'description': description
    }
    hazard_category_mapping[key] = category

print(f"\n[OK] Classified {len(hazard_classifications)} cases")

# Count by category
category_counts = Counter(hazard_category_mapping.values())
print(f"\n[INFO] Classification results:")
for category in sorted(category_counts.keys()):
    print(f"   {category:25s}: {category_counts[category]:6d} cases")

In [ ]:
# ============================================================
# ANALYSIS AND RESULTS
# ============================================================
print("[INFO] Performing comprehensive hazard analysis...\n")

# Create results dataframe
results_data = []
for hazard, classification in hazard_classifications.items():
    results_data.append({
        'Original_Hazard': hazard,
        'Classified_Category': classification['category'],
        'Confidence': classification['confidence'],
        'Hazard_Length': len(str(hazard))
    })

results_df = pd.DataFrame(results_data)

# Save results as JSON
results_json_file = os.path.join(RESULTS_DIR, 'hazard_classifications.json')
with open(results_json_file, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f"[OK] Results saved to {results_json_file}")

# Analysis by category
print(f"\n[INFO] Hazard Category Distribution:")
print(results_df['Classified_Category'].value_counts())

# Statistics
print(f"\n[INFO] Statistics:")
print(f"   Total unique hazards: {len(results_df)}")
print(f"   Average hazard text length: {results_df['Hazard_Length'].mean():.0f} chars")
print(f"   Median hazard text length: {results_df['Hazard_Length'].median():.0f} chars")
print(f"   Average confidence: {results_df['Confidence'].mean():.3f}")


In [ ]:
# ============================================================
# SAVE COMPREHENSIVE MAPPING
# ============================================================
print("[INFO] Saving comprehensive hazard mapping...\n")

# Save pickled mapping for future use
mapping_file = os.path.join(RESULTS_DIR, 'hazard_category_mapping.pkl')
with open(mapping_file, 'wb') as f:
    pickle.dump(hazard_category_mapping, f)
print(f"[OK] Mapping saved to {mapping_file}")

# Create summary report
summary = {
    'iteration': 9,
    'research_question': 'RQ3 - Hazard Type Classification',
    'total_unique_hazards': len(unique_hazards),
    'hazard_categories_identified': list(category_counts.keys()),
    'category_distribution': dict(category_counts),
    'model_used': MODEL_NAME,
    'device_used': str(device),
    'average_confidence': float(results_df['Confidence'].mean())
}

summary_file = os.path.join(RESULTS_DIR, 'iteration_9_summary.json')
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"[OK] Summary saved to {summary_file}")

print(f"\n[OK] Iteration 9 complete!")
print(f"[INFO] Results saved to: {RESULTS_DIR}")
# Add language code mapping and data sources to summary (consistent with Iteration 0)
if 'summary' in dir():
    summary['language_code_map'] = LANGUAGE_CODE_MAP
    summary['data_sources'] = {'master_df': os.path.basename(MASTER_DF_FILE)}
    print("[OK] Added language_code_map and data_sources to summary")


In [ ]:
# ============================================================
# BLUE-THEMED INTERACTIVE VISUALIZATIONS WITH PLOTLY (DARK BLUE BARS)
# ============================================================
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Define a consistent dark blue color
DARK_BLUE = '#08306b'

# 1. Hazard Category Distribution (Bar Chart)
display(Markdown("""
### Hazard Category Distribution
This interactive bar chart shows the number of unique hazards classified into each category. Hover for details.
"""))
cat_counts = results_df['Classified_Category'].value_counts().reset_index()
cat_counts.columns = ['Hazard Category', 'Count']
fig1 = px.bar(
    cat_counts,
    x='Hazard Category', y='Count',
    title='Hazard Category Distribution',
    text='Count',
)
fig1.update_traces(marker_color=DARK_BLUE, textposition='outside')
fig1.update_layout(
    title_font_size=20,
    xaxis_title_font=dict(size=16, color='navy'),
    yaxis_title_font=dict(size=16, color='navy'),
    plot_bgcolor='white',
    showlegend=False
)
fig1.show()

# 2. Hazard Text Length Distribution (Histogram)
display(Markdown("""
### Hazard Text Length Distribution
This histogram shows the distribution of hazard description lengths (in characters).
"""))
fig2 = px.histogram(
    results_df,
    x='Hazard_Length',
    nbins=30,
    title='Hazard Text Length Distribution',
)
fig2.update_traces(marker_color=DARK_BLUE)
fig2.update_layout(
    xaxis_title='Text Length (characters)',
    yaxis_title='Frequency',
    title_font_size=20,
    plot_bgcolor='white'
)
fig2.show()

# 3. Confidence Score Distribution (Bar Chart)
display(Markdown("""
### Confidence Score Distribution
This bar chart shows the distribution of confidence scores assigned to each hazard classification.
"""))
conf_counts = results_df['Confidence'].value_counts().sort_index().reset_index()
conf_counts.columns = ['Confidence', 'Count']
fig3 = px.bar(
    conf_counts,
    x='Confidence', y='Count',
    title='Confidence Score Distribution',
    text='Count',
)
fig3.update_traces(marker_color=DARK_BLUE, textposition='outside')
fig3.update_layout(
    xaxis_title='Confidence Score',
    yaxis_title='Count',
    title_font_size=20,
    plot_bgcolor='white',
    showlegend=False
)
fig3.show()

# 4. Hazard Category by Country (Stacked Bar)
display(Markdown("""
### Hazard Category by Country
This stacked bar chart shows the count of each hazard category per country, enabling cross-country comparison.
"""))
if 'SL_COUNTRY' in master_df.columns:
    cat_country = master_df.copy()
    cat_country['Hazard_Category'] = cat_country['HAZARD'].map(hazard_category_mapping)
    cat_country = cat_country.dropna(subset=['Hazard_Category', 'SL_COUNTRY'])
    pivot = cat_country.pivot_table(index='SL_COUNTRY', columns='Hazard_Category', values='HAZARD', aggfunc='count', fill_value=0)
    # Use dark blue for all bars
    fig4 = go.Figure()
    for col in pivot.columns:
        fig4.add_trace(go.Bar(
            x=pivot.index,
            y=pivot[col],
            name=col,
            marker_color=DARK_BLUE
        ))
    fig4.update_layout(
        barmode='stack',
        xaxis_title='Country',
        yaxis_title='Hazard Count',
        title='Hazard Category by Country',
        title_font_size=20,
        plot_bgcolor='white',
        legend_title_text='Hazard Category'
)
    fig4.show()
